# Analisis Exploratorio de Datos (EDA)

En este archivo se lleva acabo un analisis exloratorio sobre el dataset de personas desaparecidas ... TODO

**Integrantes:**  
1. José Rubén Alfaro González  
2. Daniela Sanluis
3. Edwin Rodriguez 

**Profesor:**  
Jessica Santizo Galicia

**Ayudantes:**  
1. Diego Antonio Villalba González 
2. Ares Gael Castro Romero

**Número de tarea:**  
[Tarea #02]

# Configuración e importación de librerias
Configuramos el ambiente para trabajar. Usaremos librerias como pandas para la lectura de datos

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

print("Librerias importadas exitosamente")

# Inicio del análisis

## 1. Carga del dataset
Cargamos el dataset. Además, podemos ver la forma del DS, que contiene 133,886 filas, y 11 columnas. Más información respecto a la información que proporciona cada una de las columnas será dada en el pdf que adjuntamos. 

In [ ]:
# Cargar dataset
df = pd.read_csv('../data/data_secretariado.csv')
f = df.shape[0]
col = df.shape[1]

print(f"Dataset cargado exitosamente")
print("Número de filas:", f)
print("Número de columnas:", col)

print("Primera inspección: ")
df.head()

También podemos ver los tipos de datos con los que vamos a contar. La mayoría serán datos categoricos o temporales, pero a partir de estos podemos calcular ciertos datos que nos serán de utilidad como la edad al momento de la desaparición (lo haremos más adelante).

In [ ]:
df.info()

Y la cantidad de valores únicos de cada columna. Hay que tomar en cuenta que uno de esos valores es CONFIDENCIAL

In [ ]:
def valores_unicos(df, columna):
    conteo = df[columna].value_counts().reset_index()
    conteo.columns = [columna, "Conteo"]
    conteo["Porcentaje"] = (conteo["Conteo"] / len(df) * 100).round(2)
    
    display(conteo.style
        .set_caption(f"Valores únicos en {columna}")
        .bar(subset=["Conteo"], color="#E05C5C", vmin=0)
        .format({"Conteo": "{:,}", "Porcentaje": "{:.2f}%"})
        .set_properties(**{"text-align": "left"})
    )

resumen = pd.DataFrame({
    "Columna": df.columns,
    "Valores_Unicos": [df[col].nunique() for col in df.columns]
}).sort_values("Valores_Unicos", ascending=False)

# Display hecho con ayuda de claude para mejor visualización
display(resumen.style
    .set_caption("Valores únicos por columna")
    .bar(subset=["Valores_Unicos"], color="#E05C5C", vmin=0)
    .format({"Valores_Unicos": "{:,}"})
    .set_properties(**{"text-align": "left"})
)

## 2. Análisis de datos faltantes y confidenciales
### Cantidad de datos faltantes y confidenciales
A continuación se muestra la cantidad de datos nulos del dataset. Podemos ver que la mayoría de datos nulos son de los campos de fechas, ya sea de nacimiento, desaparicion, o de registro. Además, es un porcentaje bastante alto de información perdida, llegando casi a un 60% para fechas de nacimiento, y 43% para las demás fechas. Fuera de eso, las demás columnas no tienen datos faltantes, pero falta revisar la cantidad de datos confidenciales.

También se muestra la cantidad de información confidencial. Se puede ver que la cantidad de información confidencial en cada columna es la misma, por lo que podemos intuir que son casos que siguen siendo confidenciales debido a varias razones (casos abiertos, en proceso de investigación, etc.)

In [ ]:
missing = pd.DataFrame({
    'Columna': df.columns,
    'Valores_Nulos': df.isnull().sum(),
    'Porcentaje': (df.isnull().sum() / len(df)) * 100
}).sort_values('Valores_Nulos', ascending=False)

print("Valores nulos por columna:")
print(missing.to_string(index=False))

confidential = pd.DataFrame({
    'Columna': df.columns,
    'Valores_Confidenciales': [(df[col] == 'CONFIDENCIAL').sum() for col in df.columns],
    'Porcentaje': [(df[col] == 'CONFIDENCIAL').sum() / len(df) * 100 for col in df.columns]
}).sort_values('Valores_Confidenciales', ascending=False)

print("\nValores confidenciales por columna:")
print(confidential.to_string(index=False))


In [ ]:
def faltantes_por_entidad(df, columna, contar_confidencial=True):
    resultados = []
    for entidad, sub_df in df.groupby("ENTIDAD"):
        registros = len(sub_df)
        nulos = sub_df[columna].isnull().sum()
        confidencial = (sub_df[columna] == "CONFIDENCIAL").sum() if contar_confidencial else 0
        faltantes = nulos + confidencial
        porcentaje = (faltantes / registros) * 100
        resultados.append({
            "Entidad": entidad.upper(),
            "Porcentaje": round(porcentaje, 2)
        })
    return pd.DataFrame(resultados)

Además, con un poco (bastante) de ayuda de Claude, podemos visualizar de manera más efectiva la cantidad de información faltante por cada uno de los estados y el porcentaje correspondiente. Lo que si vemos, es que la mayoría de la información perdida no es debido a que no se registre, sino que hay una gran cantidad de información confidencial. Además, no parece haber relación alguna entre datos faltantes y los estados, sin embargo, si es posible ver que hay estados como Jalisco que tienen mucha información faltante en la mayoría de las columnas. 

In [ ]:
# Cargar GeoJSON
mexico = gpd.read_file("https://raw.githubusercontent.com/angelnmara/geojson/master/mexicoHigh.json")
mexico["name"] = mexico["name"].str.upper()

# Mapeo de nombres si es necesario
mapeo = {
    "COAHUILA DE ZARAGOZA": "COAHUILA",
    "VERACRUZ DE IGNACIO DE LA LLAVE": "VERACRUZ",
    "MICHOACÁN DE OCAMPO": "MICHOACÁN",
    # agregar los que no coincidan
    "ESTADO DE MÉXICO": "MÉXICO",
}

def mostrar_mapa(columna, contar_confidencial):
    datos = faltantes_por_entidad(df, columna, contar_confidencial)
    datos["Entidad"] = datos["Entidad"].replace(mapeo)
    
    # Desglose por entidad
    desglose = []
    for entidad, sub_df in df.groupby("ENTIDAD"):
        registros = len(sub_df)
        nulos = sub_df[columna].isnull().sum()
        confidencial = (sub_df[columna] == "CONFIDENCIAL").sum()
        desglose.append({
            "Entidad": entidad.upper(),
            "Registros": registros,
            "Nulos": nulos,
            "Confidencial": confidencial,
            "Total_Faltantes": nulos + confidencial,
            "% Faltantes": round((nulos + confidencial) / registros * 100, 2)
        })
    desglose = pd.DataFrame(desglose).sort_values("% Faltantes", ascending=False)
    print(f"\nDesglose de datos faltantes en: {columna}")
    print(desglose.to_string(index=False))
    
    # Mapa
    mapa = mexico.merge(datos, left_on="name", right_on="Entidad", how="left")
    
    fig, ax = plt.subplots(1, 1, figsize=(15, 10))
    mapa.plot(
        column="Porcentaje",
        cmap="YlOrRd",
        linewidth=0.8,
        edgecolor="black",
        legend=True,
        legend_kwds={"label": "% Datos Faltantes", "shrink": 0.6},
        missing_kwds={"color": "lightgrey", "label": "Sin datos"},
        vmin=0,
        vmax=100,
        ax=ax
    )
    titulo = f"Datos faltantes en {columna} por entidad"
    titulo += " (incluye CONFIDENCIAL)" if contar_confidencial else " (solo nulos)"
    ax.set_title(titulo, fontsize=14)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

columnas = [col for col in df.columns if col not in ["ID_VICTIMA", "ENTIDAD", "CVE_ENT"]]

widgets.interact(
    mostrar_mapa,
    columna=columnas,
    contar_confidencial=widgets.Checkbox(value=True, description="Incluir CONFIDENCIAL")
)

## 3. Analisis de localización
Las variables que estaremos utilizando serán las siguientes:

EDAD_DESAPARICION: Sacaremos moda, media, mediana y cuartiles

SEXO: Sacaremos moda

ESTATUS_VICTIMA: Sacaremos moda

ENTIDAD: Sacaremos moda

MUNICIPIO: Sacaremos moda

FECHA_DESAPARICION: Sacaremos moda y mediana

En la mayoría de las variables anteriores, la información nula es mínima, sin embargo, la cantidad de información confidencial es considerable, y podría meter ruido al análisis de los datos. Además, en algunos estados la perdida de información llega incluso a 60% y en casos como los de Campeche que 60% de 135 nos deja con solamente ~50 casos con información útil, lo cual hace imposible hacer una estadistica más general para el estado. Aún así, la información confidencial no aporta más allá de lo visualizado anteriormente, y no podemos "rellenar" los datos pues no hay ni (1) suficiente información y (2) los casos confidenciales no tienen nada de información que nos permita "intuir" las demás medidad pues toda su información es confidencial. Por eso, decidimos crear un sub-dataset con la información confidencial transformada como NaN.

Ahora, también hay una probabilidad de que los casos confidenciales sean todos de un tipo en particular, o que representen a cierto grupo de los datos en particular, por lo que no tener esa información podría introducir cierto sesgo en el análisis.

In [ ]:
df_clean = df.copy()
date_columns = ['FECHA_NACIMIENTO', 'FECHA_DESAPARICION', 'FECHA_REGISTRO', 'SEXO', 'ESTATUS_VICTIMA', 'MUNICIPIO']
for col in date_columns:
    df_clean[col] = df_clean[col].replace('CONFIDENCIAL', pd.NA)

df_clean.head()


df_clean['FECHA_NACIMIENTO'] = pd.to_datetime(df_clean['FECHA_NACIMIENTO'], errors='coerce')
df_clean['FECHA_DESAPARICION'] = pd.to_datetime(df_clean['FECHA_DESAPARICION'], errors='coerce')
df_clean['FECHA_REGISTRO'] = pd.to_datetime(df_clean['FECHA_REGISTRO'], errors='coerce')

'''
Además, agregamos una nueva columna 'EDAD_DESAPARICION' que calcula la edad de la víctima al momento de su desaparición,
restando la fecha de nacimiento de la fecha de desaparición y dividiendo por 365 para obtener la edad en años.
'''
df_clean['EDAD_DESAPARICION'] = (df_clean['FECHA_DESAPARICION'] - df_clean['FECHA_NACIMIENTO']).dt.days // 365

df_clean.head()

### Funciones auxiliares

In [ ]:
'''
Funciones auxiliares para graficar
'''
def plot_histograma(series, titulo=None, color="#E05C5C", bins=40):
    """
    Grafica un histograma de una serie numérica con media y mediana.

    Args:
        series  : Serie de pandas a graficar (ya limpia, sin NaN)
        titulo  : Título de la gráfica. Si es None, usa el nombre de la serie
        color   : Color de las barras (hex o nombre)
        bins    : Número de bins del histograma
    """
    s = series.dropna()
    titulo = titulo or f"Distribución de {s.name}"

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(s, bins=bins, color=color, edgecolor="white", linewidth=0.5)
    ax.axvline(s.mean(),   color="#2C2C2C", linestyle="--", linewidth=1.5, label=f"Media: {s.mean():.1f}")
    ax.axvline(s.median(), color="#555555", linestyle=":",  linewidth=1.5, label=f"Mediana: {s.median():.1f}")
    ax.set_title(titulo, fontsize=14, fontweight="bold", pad=12)
    ax.set_xlabel(s.name)
    ax.set_ylabel("Frecuencia")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

### EDAD_DESAPARICION
Vamos a calcular las medidas de localización de la edad de desaparición de las personas. Usando las funciones de pandas, obtener sus cosas es sencillo. Ahora, de los resultados notamos algunas cosas raras. Parece ser que hay ciertos ejemplares que fueron mal puestos, pues los extremos de los rangos son [-13, 1830], es decir, hay personas que tienen -13 años, y otras que tienen 1830.

In [ ]:
df_clean['EDAD_DESAPARICION'].describe()

In [ ]:
plot_histograma(df_clean["EDAD_DESAPARICION"], titulo="Distribución de edad al momento de desaparición")

Esto es claramente un error en el registro de los datos (a menos que haya personas de 1830 y -13 años), y queremos ver cuantos de estos hay. Asumiremos que una persona a lo más puede vivir 130 años actualmente, por lo que veremos cuantas personas hay fuera de ese rango. Podemos ver que la cantidad no es tanta, sin embargo, cuando reducimos el rango a 100 años como máximo, la cantidad de personas se cuatriplica. Esto podría significar que efectivamente hay ~400 personas que desaparecieron con más de 100 años, pero también podría significar un error de dedo al teclear las fechas.

In [ ]:
fuera_rango = df_clean[(df_clean["EDAD_DESAPARICION"] < 0) | (df_clean["EDAD_DESAPARICION"] > 130)]

print("Registros fuera del rango [0, 130]:", len(fuera_rango))
print("Porcentaje del total:", round(len(fuera_rango) / len(df) * 100, 2))

In [ ]:
df_rangVal = df_clean[
    (df_clean["EDAD_DESAPARICION"] >= 0) & 
    (df_clean["EDAD_DESAPARICION"] <= 130)
].copy()

df_rangVal['EDAD_DESAPARICION'].describe()

In [ ]:
plot_histograma(df_rangVal['EDAD_DESAPARICION'], titulo="Edad al momento de desaparición (solo valores válidos)")

### SEXO
Ahora, calcularemos la moda del sexo de las personas desaparecidas. Antes de cualqueir cosa, cuando calculamos la moda sobre el dataset original, la moda es CONFIDENCIAL, por lo que el calculo lo haremos sobre el dataset "limpio"

In [ ]:
df_clean['SEXO'].mode()

La mayor cantidad de personas desaparecidas son hombres. Ahora, si queremos tratar de ver más allá de esto, no podemos descartar la posibilidad que dentro de los registros, se mantenga como CONFIDENCIAL todos aquellos casos que se catalogen como posible secuetro por género (similar al delito de feminicidio), y que por lo tanto la moda se vea afectada por esto.

Algunos análisis que podríamos llevar acabo sería ver cual es la moda por estado, y buscar si por ejemplo, en algún estado en paricular donde la moda sea mujer, si hay algún problema relacionado al trato de personas y particularmente mujeres, o en algún estado con moda hombres, haya problemas por reclutamiento forzado del crimen organizado.

### ESTATUS_VICTIMA
Haremos exactamente el mismo procedimiento que realizamos para el sexo del dataset

In [ ]:
df_clean['ESTATUS_VICTIMA'].mode()

En este caso, tampoco hay mucho que se pueda explicar. Simplemente la mayoría de las personas desaparecidas siguen desaparecidas al día de hoy (el dataset)

### ENTIDAD
Vamos a calcular la moda de la entidad con más desapariciones.

In [ ]:
df_clean['ENTIDAD'].mode()

Esta medida es particular. Podemos ver que si nos guiamos por la cantidad de elementos "eliminados" por ser confidenciales/nulos, la moda debería de ser el Estado de México, pues es el estado que tiene el mayor número de información disponible. Sin embargo, esta es una de las medidas que no se ve afectada por la falta de información. Todas las filas tienen un valor en ENTIDAD válido, por lo que aunque no sepamos acerca de la información precisa del casos de desaparición, al menos tenemos conocimiento del estado del que se reporto, por lo que no perdemos información por confidencialidad.

### MUNICIPIO
Para esta medida sacaremos la moda, sin embargo, esta medida puede no resultar de gran utilidad al realizar el analisis en todo el dataset en lugar de hacerlo por entidad, debido a que podría sucede que por ejemplo, el municipio donde más haya desapariciones sea en un estado con la menor cantidad de desapariciones, lo cual a su vez podría suceder debido a que ENTIDAD no tiene información confidencial, pero los MUNICIPIO si, por lo que la relación que podríamos buscar se podría ver sesgada por eso.

In [ ]:
df_clean['MUNICIPIO'].mode()

Aún así, la moda resultó ser municipio desconocido.

### FECHA_DESAPARICION
Para la fecha de desaparicion sacaremos la moda y la mediana. Además, podemos gráficar la cantidad de desapariciones por año para ver una tendencia respecto al tiempo que ha pasado entre la primera desaparicion y la última. 

In [ ]:
fecha = df_clean["FECHA_DESAPARICION"].dropna()

mediana = fecha.sort_values().iloc[len(fecha) // 2]
moda = fecha.mode().iloc[0]

print("Mediana de fecha de desaparición:", mediana.date())
print("Moda de fecha de desaparición:", moda.date())

De ver la mediana, nos damos cuenta que al menos 50% de las desapariciones registradas en este dataset han sido registradas a partir del 2018, a pesar de que el día con más desapariciones fue en 1974. Esto podría ser por la precision con la que estamos calculando la moda. En la gráfica de abajo vemos que el año con más desapariciones fue 2024 (aprox), por lo que si hiciermos una moda por año, el resultado sería distinto. Hacer la moda por día no resulta muy útil, y además puede llevar a malinterpretar los resultados. Si no sacaramos medaiana, y solamente nos guiaramos por la moda del día, pensaríamos que la inseguridad es menor acutalmente, a pesar de que la gráfica de abajo muestra que la tendencía de desapariciones va en aumento.

In [ ]:
def plot_desapariciones_por_anio(df):
    por_anio = (
        df["FECHA_DESAPARICION"].dropna()
        .dt.year
        .value_counts()
        .sort_index()
        .reset_index()
    )
    por_anio.columns = ["Año", "Conteo"]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(por_anio["Año"], por_anio["Conteo"], color="#E05C5C", edgecolor="white", linewidth=0.5)
    ax.set_title("Desapariciones por año", fontsize=14, fontweight="bold", pad=12)
    ax.set_xlabel("Año")
    ax.set_ylabel("Frecuencia")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_desapariciones_por_anio(df_clean)

## 4. Analisis de variabilidad
Las medidas que estaremos utilizando serán las siguientes:

EDAD_DESAPARICION: Sacaremos varianza, rango, IQR? y desviación estandar

FECHA_DESAPARICION: Sacaremos el minimo y máximo, para así obtener el intervalo temporal

FECHA_REGISTRO: Igualmente sacaremos el rango

Como no tenemos variables númericas además de la edad, hacer un análisis de variabilidad de los datos resulta complicado. Sin embargo, podemos realizar un buen analisis sobre la edad de desaparición. 

## EDAD_DESAPARICION
Para la edad de desaparición, sacaremos la varianza, el rango, el IQR, y la desviación estandar. Primero lo haremos con el dataset con el rango sin ajustar, y luego lo haremos con el rango ajustado. Debido a que los valores son extremos, deberíamos de ver un cambio en la mayoría de las medidas.

In [ ]:
aux = df_clean['EDAD_DESAPARICION'].dropna()

var = aux.var()
desv = aux.std()
rango = aux.max() - aux.min()
iqr = aux.quantile(0.75) - aux.quantile(0.25)

print(f"Varianza: {var:.2f}")
print(f"Desviación estándar: {desv:.2f}")
print(f"Rango: {rango:.2f}")
print(f"Rango intercuartílico (IQR): {iqr:.2f}")


In [ ]:
aux = df_rangVal['EDAD_DESAPARICION'].dropna()

var = aux.var()
desv = aux.std()
rango = aux.max() - aux.min()
iqr = aux.quantile(0.75) - aux.quantile(0.25)

print(f"Varianza: {var:.2f}")
print(f"Desviación estándar: {desv:.2f}")
print(f"Rango: {rango:.2f}")
print(f"Rango intercuartílico (IQR): {iqr:.2f}")

Podemos observar que si hubo un cambio medianamente grande. Ver un cambio de aproximadamente 5 años en la desviación estandar significa que podemos ajustar de mejor forma la cantidad de casos y buscar una generalidad. Como la desviación es de 15.81 años, y el promedio es de ~32 años, significa que la mayoría de los casos de desapariciones estan en el rango de [17 - 45] años de edad. Eso significa que los dos grupos más afectados por las desapariciones son los jovenes y los adultos, pero eso forma gran parte de la población, por lo que no podríamos decir que este desapareciendo un cierto perfil de edad.

### FECHA_DESAPARICION
Para la fecha de desaparición podemos sacar un rango de valores, pero eso ya lo calculamos e hicimos un breve análisis en la sección anterior, por lo que aquí no lo pondremos

### FECHA_REGISTRO
Sacaremos el rango de valores de las fechas de registro de incidentes. Recordemos que anteriormente calculamos las edades al momento de la desaparición usando la de nacimiento y registro. Obtuvimos valores que no tenían sentido, y vimos que en muchos casos esos valores eran por fechas de nacimeinto mal escritas. Igualmente, podría ser que haya fechas de registro que este causando un cálculo erroneo de la edad de las personas, por lo que eso lo podremos ver en el rango.

In [ ]:
rango = df_clean['FECHA_REGISTRO'].max() - df_clean['FECHA_REGISTRO'].min()

print(f"Fecha mínima de registro: {df_clean['FECHA_REGISTRO'].min().date()}")
print(f"Fecha máxima de registro: {df_clean['FECHA_REGISTRO'].max().date()}")
print(f"Rango de fechas de registro: {rango.days / 365} años")

En este caso no vemos valores anormales, lo que se puede deber a que el registro de las fechas se hace de forma automática en el sistema, evitando errores como los de las fechas de nacimiento. Además, vemos que el dataset cubre las desapariciones de aproximadamente 65 años.

### Medidas de heterogeneidad
SEXO: Sacaremos la entropía de Shannon, y el índice de Simpson

ESTATUS_VICTIMA: Sacaremos la entropía de Shannon

ENTIDAD: Sacaremos la entropía de Shannon

MUNICIPIO: Sacaremos la entropía de Shannon

ORIGEN_REPORTE: Sacaremos la entropía de Shannon


### Medidas de concentración
SEXO: Sacaremos la frecuencia relativa a la moda

ENTIDAD: Sacaremos el HHI

MUNICIPIO: Sacaremos el HHI

EDAD_DESAPARICION: Sacaremos el índice de Gini y la curva de Lorenz